In [1]:
from sklearn.cluster import DBSCAN
import numpy as np
import torch
import logging
import pandas as pd
import plotly.express as px
import matplotlib.pyplot as plt

from ipywidgets import interact, Dropdown, IntSlider, SelectMultiple
from types import SimpleNamespace

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors

try:
    import umap
except Exception:
    umap = None

from vla.models.smolvla import SmolVLAPolicy
from vla.models.world_model import WorldModelEncoder, build_world_model 

from vla.utils import get_device, seed_everything
from vla.constants import DistanceMetric, LiberoSuite, Mode, WorldModelType
from vla.rl.rollout import RolloutEngine, Trajectory
from vla.rl.srpo_reward import MultiTaskWorldProgressReward, SRPORewardConfig
from vla.data.libero import LiberoSFTDataset
from vla.utils.tensor import to_float01

In [2]:
SEED = 42

device = get_device()
seed_everything(SEED)
logger = logging.getLogger(__name__)

In [6]:
checkpoint = "HuggingFaceVLA/smolvla_libero"
world_model_type:WorldModelType = "vjepa2"

num_demos = 100
task_id = 5
libero_suite = "spatial"

dbscan_eps = 0.5
dbscan_min_samples = 2
distance_metric: DistanceMetric = DistanceMetric.NORMALIZED_L2
subsample_every: int = 5
dbscan_auto_eps = False

## Collect trajectories


In [ ]:
demo_trajectories = {}
task_key = f"{libero_suite}_task_{task_id}"
task_dataset = LiberoSFTDataset(
    suite=libero_suite,
    num_demos=num_demos,
    seed=SEED,
    task_id=task_id,
)
trajs = task_dataset.episodes_as_trajectories(task_id=task_id)
for traj in trajs:
    traj.task_id = task_key
demo_trajectories[task_key] = trajs
print(f"Collected {len(demo_trajectories[task_key] )}")

Resolving data files:   0%|          | 0/34 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/34 [00:00<?, ?it/s]

Converting episodes to trajectories: 100%|██████████| 43/43 [00:08<00:00,  4.96it/s]

Collected 43


: 

In [7]:
world_encoder = build_world_model(model_type=world_model_type, device=str(device))
reward_cfg = SRPORewardConfig(
    subsample_every=subsample_every,
    dbscan_eps=dbscan_eps,
    dbscan_min_samples=dbscan_min_samples,
    distance_metric=distance_metric,
    dbscan_auto_eps=dbscan_auto_eps,
    use_failure_rewards=True,
)
reward_model = MultiTaskWorldProgressReward(world_encoder, reward_cfg)

In [8]:
for tid, demos in demo_trajectories.items():
    demo_images = []
    for dt in demos:
        imgs = dt.images[: dt.length]
        imgs = to_float01(imgs)
        demo_images.append(imgs)
    reward_model.add_demo_trajectories(tid, demo_images)

In [9]:
embeddings = reward_model._per_task[task_key]._demo_embeddings
ref_matrix = torch.stack(embeddings, dim=0)
X = ref_matrix.cpu().numpy()

db = DBSCAN(eps=20, min_samples=dbscan_min_samples, metric="euclidean").fit(X)

labels = db.labels_

unique_labels = set(labels)
unique_labels.discard(-1)

print(f"Found {len(unique_labels)} clusters")
if len(unique_labels) != 0:
    centres = []
    for label in sorted(unique_labels):
        mask = labels == label
        centres.append(ref_matrix[mask].mean(dim=0))
    cluster_centers = torch.stack(centres, dim=0)


Found 6 clusters


In [ ]:
# model = SmolVLAPolicy(checkpoint, action_dim=7)